In [ ]:
!pip install -q transformers sentencepiece x-transformers optuna scikit-learn iterative-stratification

# Experiment 2 — Prachatai `body_text` (official repository benchmark)

Notebook นี้ใช้ `body_text` และ test set 13,578 แถวตาม official PyThaiNLP 70/10/20 benchmark พร้อมตรวจ SHA256 ของ `train_valid_idx.pkl` และ `test_idx.pkl`.

ส่วนโครงสร้างโมเดล loss, optimizer, scheduler, epochs, batch size, threshold grid และ Optuna search space ถูกคัดลอกจาก notebook ต้นฉบับโดยไม่แก้ไข เปลี่ยนเฉพาะ input/labels/split และเพิ่ม metrics/export เพื่อใช้เขียน paper

Validation ใช้ตำแหน่งภายใน `train_valid_df` ที่ถูกต้อง จึงไม่ทับกับ train/test; train และ test membership ยังคงตรงกับ official index files.

Benchmark comparison (Macro accuracy / Macro-F1): fastText 0.9302/0.5529, LinearSVC 0.513277/0.552801, ULMFit 0.948737/0.744875, USE 0.856091/0.696172. ตารางนี้มาจาก repository benchmark ไม่ใช่บทความ peer-reviewed.

Reference: https://github.com/PyThaiNLP/prachathai-67k


In [ ]:
import os
import random
import hashlib
import pickle
import urllib.request
import warnings
import zipfile
from ast import literal_eval
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from tqdm import tqdm

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup
)

from x_transformers import Decoder
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

from sklearn.metrics import (
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
    classification_report,
    hamming_loss
)

import optuna


In [ ]:
SEED = 42
SPLIT_SEED = 1412

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("DEVICE:", device)

In [ ]:
MODEL_NAME = "airesearch/wangchanberta-base-att-spm-uncased"

MAX_LEN = 128

EPOCHS = 100

BATCH_SIZE = 64

NUM_LABELS = 12

N_TRIALS = 10

LABEL_COLUMNS = [
    "politics",
    "human_rights",
    "quality_of_life",
    "international",
    "social",
    "environment",
    "economics",
    "culture",
    "labor",
    "national_security",
    "ict",
    "education"
]

THAI_LABEL_COLUMNS = {
    "politics": "การเมือง",
    "human_rights": "สิทธิมนุษยชน",
    "quality_of_life": "คุณภาพชีวิต",
    "international": "ต่างประเทศ",
    "social": "สังคม",
    "environment": "สิ่งแวดล้อม",
    "economics": "เศรษฐกิจ",
    "culture": "วัฒนธรรม",
    "labor": "แรงงาน",
    "national_security": "ความมั่นคง",
    "ict": "ไอซีที",
    "education": "การศึกษา"
}

TEXT_COLUMN = "body_text"
EXPECTED_TOTAL_ROWS = 67889
EXPECTED_SPLIT_SIZES = (47522, 6789, 13578)

RAW_FILE_NAMES = [
    "prachathai-67k.csv",
    "prachatai-67k.csv",
    "prachathai_67k.csv",
    "prachatai_67k.csv"
]

SEARCH_ROOTS = [
    Path("/content"),
    Path("/content/prachatai_official_data"),
    Path("/mnt/data")
]

# Exact reference indices committed by PyThaiNLP.
INDEX_FILES = {
    "train_valid_idx.pkl": {
        "url": "https://raw.githubusercontent.com/PyThaiNLP/prachathai-67k/master/train_valid_idx.pkl",
        "sha256": "5cc99196edf9637ed435717c01a9b5a34c0083ae986d0884f0a9530b32894ddf"
    },
    "test_idx.pkl": {
        "url": "https://raw.githubusercontent.com/PyThaiNLP/prachathai-67k/master/test_idx.pkl",
        "sha256": "61c90243c6b1b1c4bd68df8ea6df447c2886c88ad2a57cd724aa03be73e13897"
    }
}

INDEX_DIR = Path("/content/prachatai_official_indices")
AUTO_DOWNLOAD_OFFICIAL_RAW = True
OFFICIAL_RAW_URL = "https://www.dropbox.com/s/fsxepdka4l2pr45/prachathai-67k.zip?dl=1"
OFFICIAL_RAW_ZIP = Path("/content/prachathai-67k-official.zip")
OFFICIAL_RAW_DIR = Path("/content/prachatai_official_data")


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file_obj:
        for chunk in iter(lambda: file_obj.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def find_raw_csv():
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue

        for file_name in RAW_FILE_NAMES:
            candidates = sorted(root.rglob(file_name))
            if candidates:
                return str(candidates[0])

    return None


def ensure_official_raw_csv():
    raw_path = find_raw_csv()
    if raw_path is not None:
        return raw_path

    if not AUTO_DOWNLOAD_OFFICIAL_RAW:
        raise FileNotFoundError(
            "body_text benchmark ต้องใช้ไฟล์ดิบ prachathai-67k.csv "
            "ที่เรียงตามต้นฉบับ ไม่สามารถรวมสาม Hugging Face splits แทนได้"
        )

    print("DOWNLOADING OFFICIAL RAW DATA...")
    urllib.request.urlretrieve(OFFICIAL_RAW_URL, OFFICIAL_RAW_ZIP)
    OFFICIAL_RAW_DIR.mkdir(parents=True, exist_ok=True)

    try:
        with zipfile.ZipFile(OFFICIAL_RAW_ZIP) as archive:
            archive.extractall(OFFICIAL_RAW_DIR)
    except zipfile.BadZipFile as error:
        raise RuntimeError(
            "ดาวน์โหลดไฟล์ดิบอัตโนมัติไม่สำเร็จ กรุณาอัปโหลด "
            "prachathai-67k.csv ไปยัง /content แล้วรันเซลล์นี้ใหม่"
        ) from error

    raw_path = find_raw_csv()
    if raw_path is None:
        raise FileNotFoundError(
            "แตกไฟล์แล้วแต่ไม่พบ prachathai-67k.csv"
        )

    return raw_path


def ensure_reference_indices():
    INDEX_DIR.mkdir(parents=True, exist_ok=True)
    paths = {}

    for file_name, metadata in INDEX_FILES.items():
        path = INDEX_DIR / file_name

        if not path.exists() or sha256_file(path) != metadata["sha256"]:
            print("DOWNLOADING:", file_name)
            urllib.request.urlretrieve(metadata["url"], path)

        actual_hash = sha256_file(path)
        if actual_hash != metadata["sha256"]:
            raise ValueError(
                f"SHA256 ของ {file_name} ไม่ตรง official reference: {actual_hash}"
            )

        paths[file_name] = path

    return paths


RAW_CSV_PATH = ensure_official_raw_csv()
INDEX_PATHS = ensure_reference_indices()

print("MODEL/TRAINING LOCK")
print("MODEL_NAME:", MODEL_NAME)
print("MAX_LEN / EPOCHS / BATCH_SIZE:", MAX_LEN, EPOCHS, BATCH_SIZE)
print("N_TRIALS:", N_TRIALS)
print("FULL RAW CSV:", RAW_CSV_PATH)
print("OFFICIAL INDICES:", INDEX_PATHS)


In [ ]:
required_columns = [TEXT_COLUMN] + LABEL_COLUMNS


def parse_raw_labels(value):
    if isinstance(value, (list, tuple, set)):
        return set(value)

    if pd.isna(value):
        return set()

    try:
        parsed = literal_eval(str(value))
    except (ValueError, SyntaxError):
        parsed = [str(value)]

    if not isinstance(parsed, (list, tuple, set)):
        parsed = [parsed]

    return set(parsed)


def load_full_prachatai(path):
    available_columns = pd.read_csv(path, nrows=0).columns.tolist()

    if all(col in available_columns for col in required_columns):
        df = pd.read_csv(
            path,
            usecols=required_columns,
            low_memory=False
        )
    elif TEXT_COLUMN in available_columns and "labels" in available_columns:
        raw_df = pd.read_csv(
            path,
            usecols=[TEXT_COLUMN, "labels"],
            low_memory=False
        )

        parsed_labels = raw_df["labels"].map(parse_raw_labels)
        df = raw_df[[TEXT_COLUMN]].copy()

        for english_label, thai_label in THAI_LABEL_COLUMNS.items():
            df[english_label] = parsed_labels.map(
                lambda labels: int(thai_label in labels)
            )
    else:
        raise ValueError(
            "ไฟล์ดิบต้องมี body_text + labels หรือ body_text + "
            f"คอลัมน์ 0/1 ทั้ง 12 labels; แต่พบ {available_columns}"
        )

    df = df[required_columns].copy()
    df[TEXT_COLUMN] = df[TEXT_COLUMN].astype(str).str.strip()

    for col in LABEL_COLUMNS:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        ).fillna(0).astype(int)

        invalid_values = ~df[col].isin([0, 1])
        if invalid_values.any():
            values = df.loc[invalid_values, col].unique().tolist()
            raise ValueError(f"{col} มีค่านอกเหนือจาก 0/1: {values}")

    if len(df) != EXPECTED_TOTAL_ROWS:
        raise ValueError(
            f"ไฟล์ดิบต้องมี {EXPECTED_TOTAL_ROWS:,} แถว "
            f"แต่พบ {len(df):,} แถว"
        )

    df = df.reset_index(drop=True)
    df["_source_row_id"] = np.arange(len(df))
    return df


all_df = load_full_prachatai(RAW_CSV_PATH)

with open(INDEX_PATHS["train_valid_idx.pkl"], "rb") as file_obj:
    train_valid_idx = np.asarray(pickle.load(file_obj), dtype=np.int64)

with open(INDEX_PATHS["test_idx.pkl"], "rb") as file_obj:
    test_idx = np.asarray(pickle.load(file_obj), dtype=np.int64)

assert len(train_valid_idx) == 54311
assert len(test_idx) == 13578
assert set(train_valid_idx).isdisjoint(set(test_idx))
assert len(set(train_valid_idx) | set(test_idx)) == len(all_df)

train_valid_df = all_df.iloc[train_valid_idx].copy()
test_df = all_df.iloc[test_idx].copy().reset_index(drop=True)

# Same iterative stratification and seed as the reference notebook.
ohe_df = train_valid_df[LABEL_COLUMNS]
splitter = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.125,
    random_state=SPLIT_SEED
)

train_positions, validation_positions = next(
    splitter.split(ohe_df, ohe_df)
)

train_df = train_valid_df.iloc[train_positions].copy().reset_index(drop=True)

# The reference notebook accidentally used all_df.iloc[valid_idx].
# This corrected line keeps validation disjoint without changing official train/test membership.
val_df = train_valid_df.iloc[validation_positions].copy().reset_index(drop=True)

train_ids = set(train_df["_source_row_id"])
validation_ids = set(val_df["_source_row_id"])
test_ids = set(test_df["_source_row_id"])

assert train_ids.isdisjoint(validation_ids)
assert train_ids.isdisjoint(test_ids)
assert validation_ids.isdisjoint(test_ids)
assert len(train_ids | validation_ids | test_ids) == len(all_df)

actual_split_sizes = (len(train_df), len(val_df), len(test_df))
assert actual_split_sizes == EXPECTED_SPLIT_SIZES

print("FEATURE      :", TEXT_COLUMN)
print("SPLIT SEED   :", SPLIT_SEED)
print("TRAIN        :", len(train_df))
print("VALIDATION   :", len(val_df))
print("TEST         :", len(test_df))
print("INDEX CHECK  : PASSED — exact official train_valid/test index files")
print("LEAKAGE CHECK: PASSED — corrected disjoint validation mapping")


In [ ]:
print("TRAIN LABEL DISTRIBUTION")
print(train_df[LABEL_COLUMNS].sum())

print("\nVALIDATION LABEL DISTRIBUTION")
print(val_df[LABEL_COLUMNS].sum())

print("\nTEST LABEL DISTRIBUTION")
print(test_df[LABEL_COLUMNS].sum())

print("\nLABEL CARDINALITY")

print("TRAIN")
print(
    train_df[LABEL_COLUMNS]
    .sum(axis=1)
    .value_counts()
    .sort_index()
)

print("\nVALIDATION")
print(
    val_df[LABEL_COLUMNS]
    .sum(axis=1)
    .value_counts()
    .sort_index()
)

print("\nTEST")
print(
    test_df[LABEL_COLUMNS]
    .sum(axis=1)
    .value_counts()
    .sort_index()
)


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
class EmotionDataset(Dataset):

    def __init__(self, df):

        self.texts = df[TEXT_COLUMN].tolist()

        self.labels = df[LABEL_COLUMNS].values.astype(np.float32)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        text = str(self.texts[idx])

        labels = self.labels[idx]

        encoding = tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(labels, dtype=torch.float)
        }
train_dataset = EmotionDataset(train_df)

val_dataset = EmotionDataset(val_df)

test_dataset = EmotionDataset(test_df)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [ ]:
# pos_weight สำหรับ BCEWithLogitsLoss
# สูตรที่ใช้: negative / positive
# เหมาะกับ multi-label มากกว่า total / positive

label_counts = train_df[LABEL_COLUMNS].sum().values.astype(np.float32)

total_samples = len(train_df)

neg_counts = total_samples - label_counts

pos_weights = neg_counts / (label_counts + 1e-6)

pos_weights = torch.tensor(
    pos_weights,
    dtype=torch.float
).to(device)

print("LABEL COUNTS:", label_counts)
print("POS WEIGHTS :", pos_weights)


In [ ]:
class AttentionPooling(nn.Module):

    def __init__(self, hidden_size):

        super().__init__()

        self.attention = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, 1)
        )

    def forward(self, x, mask):

        scores = self.attention(x).squeeze(-1)

        scores = scores.masked_fill(mask == 0, -1e9)

        weights = torch.softmax(scores, dim=1)

        pooled = torch.sum(
            x * weights.unsqueeze(-1),
            dim=1
        )

        return pooled

In [ ]:
class EmotionModel(nn.Module):

    def __init__(
        self,
        depth=1,
        heads=2,
        attn_dropout=0.2,
        ff_dropout=0.2,
        ff_mult=2
    ):

        super().__init__()

        self.encoder = AutoModel.from_pretrained(
            MODEL_NAME
        )

        # Fixed Thai encoder:
        # freeze WangchanBERTa แล้ว train เฉพาะ decoder + pooling + classifier
        for param in self.encoder.parameters():
            param.requires_grad = False

        hidden_size = self.encoder.config.hidden_size

        self.decoder = Decoder(
            dim=hidden_size,
            depth=depth,
            heads=heads,
            attn_dropout=attn_dropout,
            ff_dropout=ff_dropout,
            ff_mult=ff_mult
        )

        self.pooling = AttentionPooling(hidden_size)

        self.dropout = nn.Dropout(0.4)

        self.fc = nn.Linear(
            hidden_size,
            NUM_LABELS
        )

    def forward(self, input_ids, attention_mask):

        # Encoder ถูก freeze จึงไม่ต้องเก็บ gradient
        with torch.no_grad():
            outputs = self.encoder(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

        x = outputs.last_hidden_state

        x = self.decoder(x)

        x = self.pooling(x, attention_mask)

        x = self.dropout(x)

        logits = self.fc(x)

        return logits


In [ ]:
def optimize_thresholds(y_true, y_probs):

    best_thresholds = []

    for i in range(NUM_LABELS):

        best_thr = 0.5
        best_f1 = 0

        for thr in np.arange(0.1, 0.9, 0.05):

            preds = (y_probs[:, i] >= thr).astype(int)

            score = f1_score(
                y_true[:, i],
                preds,
                zero_division=0
            )

            if score > best_f1:

                best_f1 = score

                best_thr = thr

        best_thresholds.append(best_thr)

    return np.array(best_thresholds)

In [ ]:
def evaluate(
    model,
    loader,
    thresholds=None,
    show_progress=False,
    progress_desc="EVALUATION"
):

    model.eval()

    all_labels = []
    all_probs = []

    with torch.no_grad():

        batch_iterator = (
            tqdm(loader, desc=progress_desc)
            if show_progress
            else loader
        )

        for batch in batch_iterator:

            input_ids = batch["input_ids"].to(device)

            attention_mask = batch["attention_mask"].to(device)

            labels = batch["labels"].to(device)

            logits = model(
                input_ids,
                attention_mask
            )

            probs = torch.sigmoid(logits)

            all_probs.append(
                probs.cpu().numpy()
            )

            all_labels.append(
                labels.cpu().numpy()
            )

    all_probs = np.vstack(all_probs)

    all_labels = np.vstack(all_labels)

    if thresholds is None:

        thresholds = np.array(
            [0.5] * NUM_LABELS
        )

    preds = (
        all_probs >= thresholds
    ).astype(int)

    micro_f1 = f1_score(
        all_labels,
        preds,
        average="micro",
        zero_division=0
    )

    macro_f1 = f1_score(
        all_labels,
        preds,
        average="macro",
        zero_division=0
    )

    weighted_f1 = f1_score(
        all_labels,
        preds,
        average="weighted",
        zero_division=0
    )

    samples_f1 = f1_score(
        all_labels,
        preds,
        average="samples",
        zero_division=0
    )

    micro_precision = precision_score(
        all_labels,
        preds,
        average="micro",
        zero_division=0
    )

    macro_precision = precision_score(
        all_labels,
        preds,
        average="macro",
        zero_division=0
    )

    micro_recall = recall_score(
        all_labels,
        preds,
        average="micro",
        zero_division=0
    )

    macro_recall = recall_score(
        all_labels,
        preds,
        average="macro",
        zero_division=0
    )

    # Multi-label accuracy_score is subset accuracy / exact match.
    subset_accuracy = accuracy_score(
        all_labels,
        preds
    )

    h_loss = hamming_loss(
        all_labels,
        preds
    )

    # PyThaiNLP's body_text table averages binary accuracy over 12 labels.
    per_label_accuracy = (all_labels == preds).mean(axis=0)
    macro_label_accuracy = per_label_accuracy.mean()

    return {
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "samples_f1": samples_f1,
        "micro_precision": micro_precision,
        "macro_precision": macro_precision,
        "micro_recall": micro_recall,
        "macro_recall": macro_recall,
        "accuracy": subset_accuracy,
        "subset_accuracy": subset_accuracy,
        "macro_label_accuracy": macro_label_accuracy,
        "per_label_accuracy": per_label_accuracy,
        "hamming_loss": h_loss,
        "labels": all_labels,
        "preds": preds,
        "probs": all_probs
    }


In [ ]:
def train_model(
    model,
    train_loader,
    val_loader,
    lr,
    weight_decay
):

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=pos_weights
    )

    # train เฉพาะ parameter ที่ requires_grad=True
    # encoder ถูก freeze แล้ว จึง update เฉพาะ decoder + pooling + classifier
    trainable_params = [p for p in model.parameters() if p.requires_grad]

    optimizer = torch.optim.AdamW(
        trainable_params,
        lr=lr,
        weight_decay=weight_decay
    )

    total_steps = len(train_loader) * EPOCHS

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(
            0.1 * total_steps
        ),
        num_training_steps=total_steps
    )

    use_amp = device.type == "cuda"

    scaler = torch.cuda.amp.GradScaler(
        enabled=use_amp
    )

    for epoch in range(EPOCHS):

        model.train()

        total_loss = 0

        loop = tqdm(train_loader)

        for batch in loop:

            input_ids = batch["input_ids"].to(device)

            attention_mask = batch["attention_mask"].to(device)

            labels = batch["labels"].to(device)

            optimizer.zero_grad()

            with torch.cuda.amp.autocast(
                enabled=use_amp
            ):

                logits = model(
                    input_ids,
                    attention_mask
                )

                loss = criterion(
                    logits,
                    labels
                )

            scaler.scale(loss).backward()

            torch.nn.utils.clip_grad_norm_(
                trainable_params,
                1.0
            )

            scaler.step(optimizer)

            scaler.update()

            scheduler.step()

            total_loss += loss.item()

            loop.set_description(
                f"Epoch {epoch+1}/{EPOCHS}"
            )

            loop.set_postfix(
                loss=loss.item()
            )

        val_result = evaluate(
            model,
            val_loader
        )

        print("\n======================")
        print(f"Epoch {epoch+1}")
        print("======================")

        print(
            f"VAL SUBSET ACC : {val_result['subset_accuracy']:.4f}"
        )

        print(
            f"VAL MICRO P/R/F1: "
            f"{val_result['micro_precision']:.4f} / "
            f"{val_result['micro_recall']:.4f} / "
            f"{val_result['micro_f1']:.4f}"
        )

        print(
            f"VAL MACRO P/R/F1: "
            f"{val_result['macro_precision']:.4f} / "
            f"{val_result['macro_recall']:.4f} / "
            f"{val_result['macro_f1']:.4f}"
        )

        print(
            f"VAL WEIGHTED F1: {val_result['weighted_f1']:.4f}"
        )

        print(
            f"VAL HAMMING LOSS: {val_result['hamming_loss']:.4f}"
        )

    return model


In [ ]:
def objective(trial):

    depth = trial.suggest_int(
        "depth",
        1,
        2
    )

    heads = trial.suggest_categorical(
        "heads",
        [2, 4]
    )

    attn_dropout = trial.suggest_float(
        "attn_dropout",
        0.2,
        0.5
    )

    ff_dropout = trial.suggest_float(
        "ff_dropout",
        0.2,
        0.5
    )

    ff_mult = trial.suggest_int(
        "ff_mult",
        2,
        4
    )

    lr = trial.suggest_float(
        "lr",
        1e-5,
        2e-4,
        log=True
    )

    weight_decay = trial.suggest_float(
        "weight_decay",
        1e-4,
        1e-2,
        log=True
    )

    model = EmotionModel(
        depth=depth,
        heads=heads,
        attn_dropout=attn_dropout,
        ff_dropout=ff_dropout,
        ff_mult=ff_mult
    ).to(device)

    model = train_model(
        model,
        train_loader,
        val_loader,
        lr,
        weight_decay
    )

    val_result = evaluate(
        model,
        val_loader
    )

    best_thresholds = optimize_thresholds(
        val_result["labels"],
        val_result["probs"]
    )

    val_result = evaluate(
        model,
        val_loader,
        best_thresholds
    )

    return val_result["macro_f1"]

In [ ]:
study = optuna.create_study(
    direction="maximize"
)

study.optimize(
    objective,
    n_trials=N_TRIALS
)


In [ ]:
print("\n======================")
print("BEST PARAMETERS")
print("======================")

print(study.best_params)

best_params = study.best_params

In [ ]:
final_model = EmotionModel(
    depth=best_params["depth"],
    heads=best_params["heads"],
    attn_dropout=best_params["attn_dropout"],
    ff_dropout=best_params["ff_dropout"],
    ff_mult=best_params["ff_mult"]
).to(device)

In [ ]:
final_model = train_model(
    final_model,
    train_loader,
    val_loader,
    best_params["lr"],
    best_params["weight_decay"]
)

In [ ]:
val_result = evaluate(
    final_model,
    val_loader
)

best_thresholds = optimize_thresholds(
    val_result["labels"],
    val_result["probs"]
)

print("\n======================")
print("BEST THRESHOLDS")
print("======================")

for label, thr in zip(
    LABEL_COLUMNS,
    best_thresholds
):
    print(f"{label}: {thr:.2f}")

In [ ]:
print("\nSTARTING TEST EVALUATION...")

test_result = evaluate(
    final_model,
    test_loader,
    best_thresholds,
    show_progress=True,
    progress_desc="TEST"
)

print("TEST EVALUATION COMPLETE")

print("\n======================")
print("FINAL RESULT — PRACHATAI BODY_TEXT / OFFICIAL 70-10-20 TEST")
print("======================")

print(
    f"Subset Accuracy / Exact Match : {test_result['subset_accuracy']:.4f}"
)

print(
    f"Macro Label Accuracy (benchmark): "
    f"{test_result['macro_label_accuracy']:.4f}"
)

print(
    f"Micro Precision              : {test_result['micro_precision']:.4f}"
)

print(
    f"Micro Recall                 : {test_result['micro_recall']:.4f}"
)

print(
    f"Micro F1                     : {test_result['micro_f1']:.4f}"
)

print(
    f"Macro Precision              : {test_result['macro_precision']:.4f}"
)

print(
    f"Macro Recall                 : {test_result['macro_recall']:.4f}"
)

print(
    f"Macro F1                     : {test_result['macro_f1']:.4f}"
)

print(
    f"Weighted F1                  : {test_result['weighted_f1']:.4f}"
)

print(
    f"Samples F1                   : {test_result['samples_f1']:.4f}"
)

print(
    f"Hamming Loss                 : {test_result['hamming_loss']:.4f}"
)


In [ ]:
print("\n======================")
print("CLASSIFICATION REPORT")
print("======================")

print(
    classification_report(
        test_result["labels"],
        test_result["preds"],
        target_names=LABEL_COLUMNS,
        zero_division=0
    )
)

In [ ]:
import json

SAVE_PATH = "/content/final_prachatai_body_official_wangchanberta_xtransformers.pt"
METRICS_PATH = "/content/prachatai_body_official_test_metrics.json"
COMPARISON_PATH = "/content/prachatai_body_official_benchmark_comparison.csv"

saved_test_metrics = {
    key: float(test_result[key])
    for key in [
        "subset_accuracy",
        "macro_label_accuracy",
        "micro_precision",
        "micro_recall",
        "micro_f1",
        "macro_precision",
        "macro_recall",
        "macro_f1",
        "weighted_f1",
        "samples_f1",
        "hamming_loss"
    ]
}

benchmark_results = [
    {"model": "fastText", "macro_accuracy": 0.930200, "macro_f1": 0.552900},
    {"model": "LinearSVC", "macro_accuracy": 0.513277, "macro_f1": 0.552801},
    {"model": "ULMFit", "macro_accuracy": 0.948737, "macro_f1": 0.744875},
    {"model": "USE", "macro_accuracy": 0.856091, "macro_f1": 0.696172},
    {
        "model": "WangchanBERTa + x-transformers (ours)",
        "macro_accuracy": saved_test_metrics["macro_label_accuracy"],
        "macro_f1": saved_test_metrics["macro_f1"]
    }
]

comparison_df = pd.DataFrame(benchmark_results)
comparison_df.to_csv(COMPARISON_PATH, index=False)

with open(METRICS_PATH, "w", encoding="utf-8") as metrics_file:
    json.dump(
        {
            "experiment": "prachathai67k_body_official_70_10_20",
            "split_seed": SPLIT_SEED,
            "split_sizes": {
                "train": len(train_df),
                "validation": len(val_df),
                "test": len(test_df)
            },
            "feature": TEXT_COLUMN,
            "thresholds": best_thresholds.tolist(),
            "best_params": best_params,
            "test_metrics": saved_test_metrics,
            "per_label_accuracy": {
                label: float(value)
                for label, value in zip(
                    LABEL_COLUMNS,
                    test_result["per_label_accuracy"]
                )
            }
        },
        metrics_file,
        ensure_ascii=False,
        indent=2
    )

torch.save(
    {
        "model_state_dict": final_model.state_dict(),
        "thresholds": best_thresholds,
        "best_params": best_params,
        "labels": LABEL_COLUMNS,
        "model_name": MODEL_NAME,
        "max_len": MAX_LEN,
        "feature": TEXT_COLUMN,
        "split_seed": SPLIT_SEED,
        "split_sizes": actual_split_sizes,
        "test_metrics": saved_test_metrics,
        "per_label_accuracy": test_result["per_label_accuracy"],
        "note": "Prachatai body_text; exact official train_valid/test indices; corrected disjoint validation; original model/training/Optuna code retained."
    },
    SAVE_PATH
)

print("\nBODY_TEXT BENCHMARK — MACRO ACCURACY / MACRO F1")
print(comparison_df.to_string(index=False))
print("\nMODEL SAVED     :", SAVE_PATH)
print("METRICS SAVED   :", METRICS_PATH)
print("COMPARISON SAVED:", COMPARISON_PATH)
